# Comparing interatomic potentials for dislocations in fcc Ni

**Benchmark chain**

| Step | Quantity | Tool |
|---|---|---|
| 1 | lattice parameter $a_0$ | LAMMPS (`a0.in`) |
| 2 | Burgers vectors $b$, $b_p$ | Python |
| 3 | elastic constants $C_{11}, C_{12}, C_{44}$ | LAMMPS (`elastic.in`) |
| 4 | shear modulus $\mu$, Poisson ratio $\nu$ | atomman `ElasticConstants` |
| 5 | generalized stacking-fault (γ) surface, $\gamma_{usf}$, $\gamma_{isf}$ | atomman `StackingFault` + LAMMPS (`gsf.in`) |
| 6 | equilibrium partial separation $d_{eq}$ | Python (isotropic) + atomman `Stroh` (anisotropic) |
| 7 | periodic array of dislocations (PAD) | atomman `Dislocation` + LAMMPS (`pad.in`) |
| 8 | disregistry and CRSS | atomman `disregistry`, `SDVPN` |

Run the notebook once per potential (selected in **Setup**). Each run writes `results_<potential>.json` for the final comparison.

> **Structure only.** The LAMMPS scripts in `lammps_scripts/` will be provided separately. Each section states its **script contract**: what the notebook passes in as `-var` variables, and what it reads back.

## 0. Setup
### 0.1 Install (Google Colab)
Everything comes from PyPI, so nothing needs compiling. The LAMMPS wheel is linked against MPICH, and the `[mpi]` extra installs that library. LAMMPS still runs as a **single serial process**: `mpirun` is never used.

In [ ]:
%pip install -q "lammps[mpi]==2025.7.22.4.0" "atomman==1.5.4" py3Dmol

### 0.2 Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# The LAMMPS wheel links against MPICH's libmpi.so.12, which pip installs outside the loader's search path
# (on Colab: /usr/local/lib, whose loader cache is not refreshed). Load it explicitly before importing lammps;
# also export its folder so that `!lmp ...` shell cells find it.
import os, ctypes, importlib.metadata
_libmpi = next(f.locate().resolve() for f in importlib.metadata.files('mpich') if f.name == 'libmpi.so.12')
ctypes.CDLL(str(_libmpi), mode=ctypes.RTLD_GLOBAL)
os.environ['LD_LIBRARY_PATH'] = f"{_libmpi.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}"

import lammps
from lammps import lammps as LAMMPS

import atomman as am
import atomman.unitconvert as uc

print('LAMMPS', LAMMPS(cmdargs=['-screen', 'none', '-log', 'none']).version(), '| atomman', am.__version__)

### 0.3 Tutorial files
The LAMMPS scripts (`lammps_scripts/`) and potential files (`ni_potentials/`) will be fetched from the course repository (TODO: URL).

In [ ]:
# TODO: fetch the course material, e.g.
# !git clone --depth 1 <COURSE_REPO_URL> course && cp -r course/lammps_scripts course/ni_potentials .

SCRIPT_DIR = Path('lammps_scripts')     # a0.in, elastic.in, gsf.in, pad.in
WORK       = Path('work'); WORK.mkdir(exist_ok=True)

# Let LAMMPS also find the potential files that ship with the wheel by name (e.g. Ni_u3.eam)
os.environ['LAMMPS_POTENTIALS'] = str(Path(lammps.__file__).parent / 'share' / 'lammps' / 'potentials')

### 0.4 The interatomic potential: `potential.mod`
Every LAMMPS script reads the potential from one small file with `include ${potential_file}`, after the box has been created.
**To test a different potential, edit this file (or write another one) and re-run the notebook. Nothing else changes.**

A `.mod` file holds ordinary LAMMPS commands:

| line | meaning | what to change |
|---|---|---|
| `pair_style eam/alloy` | functional form of the potential | `eam/alloy` for setfl files (`*.eam.alloy`), `eam/fs` for Finnis–Sinclair setfl files, `eam` for single-element funcfl files (`*.eam`), `meam`, `snap`, … |
| `pair_coeff * * <file> Ni` | parameter file, then one element name per atom type | the file path; the element as it is named in that file. Every structure in this notebook has **one** atom type, so there is a single `Ni`. For `pair_style eam` the element name is omitted (`pair_coeff * * Ni_u3.eam`). |
| `mass * 58.6934` | atomic mass of every type (g/mol) | only for other elements; it plays no role in static relaxations |

Examples:
```
pair_style eam                                    # Foiles, Baskes & Daw (1986), file ships with LAMMPS
pair_coeff * * Ni_u3.eam

pair_style eam/alloy                              # Mishin et al. (2002)
pair_coeff * * ni_potentials/NiAl02.eam.alloy.in Ni

pair_style meam                                   # MEAM: library file + parameter file
pair_coeff * * library.meam Ni Ni.meam Ni
```
The LAMMPS wheel used here does **not** include ML-PACE (ACE, `.yace`) or KIM. Of the machine-learned styles, SNAP and ML-IAP are available.

In [ ]:
%%writefile potential.mod
# Ni, EAM: Y. Mishin, Acta Mater. 52, 1451 (2004)
pair_style   eam/alloy
pair_coeff   * * ni_potentials/NiAl_Mishin_2004.eam.alloy.in Ni
mass         * 58.6934

In [ ]:
POTENTIAL_MOD  = Path('potential.mod')      # passed to every run_lammps() call
POTENTIAL_NAME = 'Mishin_2004'              # label for the results file
results = {'potential': POTENTIAL_NAME}
print(POTENTIAL_MOD.read_text())

**Optional: use your own potential file (Colab).** Run the cell below and pick the file(s) in the dialog. Uploaded files are saved in the current folder,
so in `potential.mod` refer to them by file name, e.g. `pair_coeff * * my_potential.eam.alloy Ni`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('uploaded:', list(uploaded))

### 0.5 Helper: running a LAMMPS script from Python
`run_lammps(script, potential_mod, variables)` starts a serial LAMMPS instance and passes the potential file and the script inputs as `-var` definitions.
It then executes the script with `L.file()`. Results are read back with `L.extract_variable()`, and structures with `system_from_lammps(L)` (0.6).

**Conventions shared by all scripts**
* Each script uses `include ${potential_file}` after the box is created, so the scripts themselves are independent of the potential.
* Scripts declare their inputs as `variable <name> index <default>`. A value passed with `-var` takes precedence, so every script can also be run on its own:
  `!lmp -in lammps_scripts/a0.in -var potential_file potential.mod -var data_file work/a0.data`.
* Units are `metal` throughout.

In [ ]:
def run_lammps(script, potential_mod, variables=None, log=None):
    script, potential_mod = SCRIPT_DIR / script, Path(potential_mod)
    for f in (script, potential_mod):
        if not f.exists():
            raise FileNotFoundError(f)
    args = ['-screen', 'none', '-log', str(log or WORK / f'{script.stem}.log'),
            '-var', 'potential_file', str(potential_mod)]
    for k, v in (variables or {}).items():
        args += ['-var', k, str(v)]
    L = LAMMPS(cmdargs=args)
    L.file(str(script))
    return L          # caller extracts results, then L.close()

### 0.6 Helpers: structures in and out of LAMMPS, and 3D views
* `system_from_lammps(L)` reads the current box and positions from a live LAMMPS instance into an atomman `System`, with atoms sorted by id.
  The result is directly comparable with the structure that was written with `read_data`.
* `cna(system)` runs a separate LAMMPS instance with `pair_style zero`, so no potential is needed, and returns the
  common-neighbour-analysis class of each atom: 1 fcc, 2 hcp, 3 bcc, 4 icosahedral, 5 other.
  The cutoff $0.854\,a_0$ lies halfway between the first and second neighbour shells of fcc.
* `show(systems, color_by=..., mask=...)` draws one or more structures side by side in linked interactive py3Dmol viewers, with the cell outlined.
  Drag to rotate, scroll to zoom.

In [ ]:
import py3Dmol
from lammps import LMP_STYLE_ATOM, LMP_TYPE_VECTOR

CNA_NAMES   = {1: 'fcc', 2: 'hcp', 3: 'bcc', 4: 'ico', 5: 'other'}
CNA_COLORS  = {'fcc': '#4daf4a', 'hcp': '#e41a1c', 'bcc': '#377eb8', 'ico': '#984ea3', 'other': '#bdbdbd'}
TYPE_COLORS = {'type 1': '#1f78b4', 'type 2': '#ff7f00'}

def system_from_lammps(L, symbol='Ni'):
    boxlo, boxhi, xy, yz, xz, pbc, _ = L.extract_box()
    lx, ly, lz = np.subtract(boxhi, boxlo)
    box = am.Box(avect=[lx, 0, 0], bvect=[xy, ly, 0], cvect=[xz, yz, lz], origin=boxlo)
    n = L.extract_global('nlocal')
    order = np.argsort(L.numpy.extract_atom('id')[:n])
    atype = np.array(L.numpy.extract_atom('type')[:n][order])
    atoms = am.Atoms(atype=atype, pos=np.array(L.numpy.extract_atom('x')[:n][order]))
    return am.System(atoms=atoms, box=box, pbc=[bool(p) for p in pbc], symbols=[symbol] * int(atype.max()))

def cna(system, cutoff=None):
    cutoff = cutoff or 0.854 * (a0 if 'a0' in globals() else 3.52)
    data = WORK / 'cna.data'
    system.dump('atom_data', f=str(data))
    L = LAMMPS(cmdargs=['-screen', 'none', '-log', 'none'])
    L.commands_list(['units metal', 'atom_style atomic',
                     'boundary ' + ' '.join('p' if p else 's' for p in system.pbc),
                     f'read_data {data}', 'mass * 1.0',
                     f'pair_style zero {cutoff + 1.0}', 'pair_coeff * *',
                     f'compute c all cna/atom {cutoff}', 'run 0'])
    n = L.extract_global('nlocal')
    ids = L.numpy.extract_atom('id')[:n]
    out = np.empty(n, dtype=int)
    out[ids - 1] = L.numpy.extract_compute('c', LMP_STYLE_ATOM, LMP_TYPE_VECTOR)[:n]
    L.close()
    return out

def _cell_edges(box):
    o, a, b, c = box.origin, box.avect, box.bvect, box.cvect
    corners = {(i, j, k): o + i * a + j * b + k * c for i in (0, 1) for j in (0, 1) for k in (0, 1)}
    return [(corners[p], corners[q]) for p in corners for q in corners
            if sum(abs(np.subtract(p, q))) == 1 and p < q]

def _hex(c):
    return c if isinstance(c, str) else '#%02x%02x%02x' % tuple(int(255 * v) for v in c[:3])

def show(systems, color_by='type', mask=None, titles=None, radius=0.8, width=900, height=420):
    # color_by: 'type', 'cna', or a list (one per system) of per-atom label arrays
    # mask:     None, or a list (one per system) of boolean arrays selecting the atoms to draw
    systems = list(systems) if isinstance(systems, (list, tuple)) else [systems]
    view = py3Dmol.view(width=width, height=height, viewergrid=(1, len(systems)), linked=True)
    for k, sysk in enumerate(systems):
        if color_by == 'type':
            labels, palette = np.array([f'type {t}' for t in sysk.atoms.atype]), TYPE_COLORS
        elif color_by == 'cna':
            labels, palette = np.array([CNA_NAMES[c] for c in cna(sysk)]), CNA_COLORS
        else:
            labels, palette = np.asarray(color_by[k]), CNA_COLORS
        sel = np.ones(sysk.natoms, bool) if mask is None else np.asarray(mask[k])
        classes = sorted(set(labels[sel]))
        tokens = dict(zip(classes, ['Ni', 'Cu', 'Fe', 'Au', 'Ag', 'Pt', 'Pd', 'Co']))   # XYZ "elements" carry the colour class
        xyz = f'{sel.sum()}\n\n' + '\n'.join(f'{tokens[l]} {x:.4f} {y:.4f} {z:.4f}'
                                             for l, (x, y, z) in zip(labels[sel], sysk.atoms.pos[sel]))
        vw = (0, k)
        view.addModel(xyz, 'xyz', viewer=vw)
        for i, (cls, tok) in enumerate(tokens.items()):
            color = _hex(palette.get(cls, plt.cm.tab10(i)))
            view.setStyle({'elem': tok}, {'sphere': {'color': color, 'radius': radius}}, viewer=vw)
        for p, q in _cell_edges(sysk.box):
            view.addLine({'start': dict(zip('xyz', map(float, p))), 'end': dict(zip('xyz', map(float, q))),
                          'color': 'black'}, viewer=vw)
        view.zoomTo(viewer=vw)
        counts = ', '.join(f'{c}: {(labels[sel] == c).sum()}' for c in classes)
        print(f"[{k + 1}] {titles[k] if titles else ''}   ({sel.sum()} atoms shown; {counts})")
    return view

## 1. Lattice parameter $a_0$
The structure is built in Python as an `NCELL`³ fcc supercell at a guessed lattice parameter. It is shown, passed to LAMMPS with `read_data`,
and relaxed at zero pressure (`fix box/relax iso 0.0`). The relaxed cell is then read back from LAMMPS and shown next to the initial one.

**Script contract - `a0.in`**

| in (`-var`) | out (`extract_variable`) |
|---|---|
| `data_file`, `ncell` | `a0` [Å] (= `lx/ncell`), `Ecoh` [eV/atom] |

In [ ]:
A_GUESS, NCELL = 3.52, 4

def fcc_ucell(a):
    return am.System(atoms=am.Atoms(atype=1, pos=[[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
                     box=am.Box.cubic(a=a), scale=True, symbols='Ni')

bulk_initial = fcc_ucell(A_GUESS).supersize(NCELL, NCELL, NCELL)
show(bulk_initial, titles=[f'initial fcc cell, a = {A_GUESS} Å']).show()

In [ ]:
bulk_initial.dump('atom_data', f=str(WORK / 'a0.data'))
L = run_lammps('a0.in', POTENTIAL_MOD, {'data_file': WORK / 'a0.data', 'ncell': NCELL})
a0, Ecoh = L.extract_variable('a0'), L.extract_variable('Ecoh')
p_res = L.get_thermo('press')
bulk_relaxed = system_from_lammps(L)
L.close()

results.update(a0=a0, Ecoh=Ecoh)
print(f'a0 = {a0:.4f} Å   Ecoh = {Ecoh:.4f} eV/atom   residual pressure = {p_res:.2e} bar')
show([bulk_initial, bulk_relaxed], color_by='cna',
     titles=[f'initial, a = {A_GUESS} Å', f'relaxed, a0 = {a0:.4f} Å']).show()

ucell = fcc_ucell(a0)      # conventional fcc cell at the relaxed a0, reused in Sections 3, 5 and 7

## 2. Burgers vectors from $a_0$
Perfect dislocation: $\mathbf b = \tfrac{a_0}{2}[1\bar10]$, $|\mathbf b| = a_0/\sqrt2$.
Shockley partials: $\mathbf b = \mathbf b_{p1} + \mathbf b_{p2} = \tfrac{a_0}{6}[2\bar1\bar1] + \tfrac{a_0}{6}[1\bar21]$, $|\mathbf b_p| = a_0/\sqrt6$.
(111) interplanar spacing: $d_{111} = a_0/\sqrt3$.

In [ ]:
b_uvw   = np.array([1, -1, 0]) / 2       # crystal (uvw) coordinates, in units of a0
bp1_uvw = np.array([2, -1, -1]) / 6
bp2_uvw = np.array([1, -2, 1]) / 6
assert np.allclose(bp1_uvw + bp2_uvw, b_uvw)

b, bp, d111 = a0 / np.sqrt(2), a0 / np.sqrt(6), a0 / np.sqrt(3)
results.update(b=b, b_p=bp, d111=d111)
print(f'|b| = {b:.4f} Å   |b_p| = {bp:.4f} Å   d111 = {d111:.4f} Å')

## 3. Elastic constants
**Script contract - `elastic.in`**

| in | out |
|---|---|
| `data_file` (fcc supercell at $a_0$) | `C11`, `C12`, `C44` [GPa] |

Cubic symmetry means three independent constants. The values are stored in an `atomman.ElasticConstants` object, which is used in every later step.

In [ ]:
ucell.supersize(NCELL, NCELL, NCELL).dump('atom_data', f=str(WORK / 'elastic.data'))
L = run_lammps('elastic.in', POTENTIAL_MOD, {'data_file': WORK / 'elastic.data'})
Cij_GPa = {k: L.extract_variable(k) for k in ('C11', 'C12', 'C44')}
L.close()

C = am.ElasticConstants(**{k: uc.set_in_units(v, 'GPa') for k, v in Cij_GPa.items()})
results.update(Cij_GPa)
print({k: round(v, 1) for k, v in Cij_GPa.items()}, 'GPa')
print(np.round(uc.get_in_units(C.Cij, 'GPa'), 1))

## 4. Shear modulus from the elastic constants
* Isotropic averages: Voigt, Reuss and Hill estimates, from `C.shear()` and `C.bulk()`. From these, $\nu = \dfrac{3K-2\mu}{2(3K+\mu)}$.
* Shear modulus on the slip system $\{111\}\langle110\rangle$: $\mu_{111} = (C_{11}-C_{12}+C_{44})/3$.
* Zener anisotropy ratio: $A = 2C_{44}/(C_{11}-C_{12})$. Ni is strongly anisotropic, with $A$ of about 2.5, which is why Section 6 also gives the anisotropic estimate.

In [ ]:
mu = {s: uc.get_in_units(C.shear(s), 'GPa') for s in ('Voigt', 'Reuss', 'Hill')}
K  = uc.get_in_units(C.bulk('Hill'), 'GPa')
nu = (3 * K - 2 * mu['Hill']) / (2 * (3 * K + mu['Hill']))
c11, c12, c44 = (Cij_GPa[k] for k in ('C11', 'C12', 'C44'))
mu111, A = (c11 - c12 + c44) / 3, 2 * c44 / (c11 - c12)

results.update(mu_Voigt=mu['Voigt'], mu_Reuss=mu['Reuss'], mu_Hill=mu['Hill'],
               K_Hill=K, nu_Hill=nu, mu_111=mu111, Zener_A=A)
for k in ('mu_Voigt', 'mu_Reuss', 'mu_Hill', 'mu_111', 'K_Hill', 'nu_Hill', 'Zener_A'):
    print(f'{k:9s} = {results[k]:8.3f}')

## 5. Generalized stacking-fault energy
atomman builds a (111) slab and generates the rigidly shifted configurations on a grid of fractional shifts $(a_1, a_2)$
along $\mathbf a_1 = \tfrac12[1\bar10]$ and $\mathbf a_2 = \tfrac12[10\bar1]$. LAMMPS relaxes each configuration perpendicular to the fault only.
The SDVPN model in Section 8 needs this full 2D γ-surface; the 1D curve along $[2\bar1\bar1]$ is a cut through it.

**Script contract - `gsf.in`** (called once per shift)

| in | out |
|---|---|
| `data_file` (atomman `atom_data`) | `E_total` [eV] |

The slab is periodic in-plane and has free surfaces along z (`boundary p p s`). Atoms are relaxed along z only (`fix setforce 0 0 NULL`).

In [ ]:
NUM_A1 = NUM_A2 = 12       # γ-surface grid (12x12 = 144 small relaxations); multiple of 6 so 1/6, 1/3, 2/3 are on-grid
SLAB_MINWIDTH = 40.0       # Å, slab thickness normal to (111)

a1_uvw, a2_uvw = np.array([1, -1, 0]) / 2, np.array([1, 0, -1]) / 2
sf = am.defect.StackingFault('1 1 1', ucell, a1vect_uvw=a1_uvw, a2vect_uvw=a2_uvw)
sf.surface(minwidth=SLAB_MINWIDTH, even=True)
print('slab atoms:', sf.system.natoms, '| fault area: %.3f Å^2' % sf.surfacearea)

In [ ]:
a1s, a2s, Es = [], [], []
for a1, a2, system in sf.iterfaultmap(num_a1=NUM_A1, num_a2=NUM_A2):
    data_file = WORK / 'gsf.data'
    system.dump('atom_data', f=str(data_file))
    L = run_lammps('gsf.in', POTENTIAL_MOD, {'data_file': data_file}, log='none')
    a1s.append(a1); a2s.append(a2); Es.append(L.extract_variable('E_total'))
    L.close()

E0 = Es[0]                               # (a1, a2) = (0, 0): perfect stacking
assert a1s[0] == 0 and a2s[0] == 0
E_gsf = (np.array(Es) - E0) / sf.surfacearea      # eV/Å^2
gamma = am.defect.GammaSurface(a1vect=a1_uvw, a2vect=a2_uvw, box=ucell.box,
                               a1=a1s, a2=a2s, E_gsf=E_gsf)

In [ ]:
fig = gamma.E_gsf_surface_plot(energyperarea_unit='mJ/m^2')
fig = gamma.E_gsf_line_plot(vect=3 * bp1_uvw, energyperarea_unit='mJ/m^2')   # 0 -> isf -> ... along [2-1-1]

In [ ]:
# b_p1 = (a1 + a2)/3. Whether +b_p1 or -b_p1 leads to the intrinsic fault depends on the stacking
# sense of the slab, so evaluate both and keep the side that ends in the lower-energy minimum (the ISF).
s = np.linspace(0, 1, 201)                                   # fraction of |b_p|
paths = {sign: uc.get_in_units(gamma.E_gsf(a1=(sign * s / 3) % 1, a2=(sign * s / 3) % 1), 'mJ/m^2')
         for sign in (+1, -1)}
sign = min(paths, key=lambda k: paths[k][-1])
g_path = paths[sign]
gamma_usf, s_usf = g_path.max(), s[g_path.argmax()]
gamma_isf = g_path[-1]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(s, g_path); ax.axhline(gamma_isf, ls=':', c='k')
ax.set(xlabel=r'shift / $|b_p|$ along ' + ('+' if sign > 0 else '-') + r'$b_{p1}$', ylabel=r'$\gamma$ (mJ/m$^2$)')
results.update(gamma_usf=gamma_usf, gamma_isf=gamma_isf)
print(f'γ_usf = {gamma_usf:.1f} mJ/m² at {s_usf:.2f} b_p  |  γ_isf = {gamma_isf:.1f} mJ/m²')

**Looking at the intrinsic stacking fault.** The slab is shown perfect and with the shift $\mp\mathbf b_{p1}$, both after relaxation, coloured by CNA.
The intrinsic stacking fault changes the stacking from …ABCABC… to …ABC|BCA…: one ABC sequence loses a layer. This leaves two (111) layers with hcp coordination (red).
Atoms at the free surfaces of the slab are classed as *other* (grey). For display only, the slab is repeated 4×4 in-plane.

In [ ]:
a_isf = (sign / 3) % 1
slabs = {}
for name, (f1, f2) in {'perfect': (0.0, 0.0), 'ISF': (a_isf, a_isf)}.items():
    sf.fault(a1=f1, a2=f2).dump('atom_data', f=str(WORK / 'gsf.data'))
    L = run_lammps('gsf.in', POTENTIAL_MOD, {'data_file': WORK / 'gsf.data'}, log='none')
    slabs[name] = system_from_lammps(L).supersize(4, 4, 1)
    L.close()
show(list(slabs.values()), color_by='cna', titles=list(slabs)).show()

## 6. Equilibrium partial separation
The two Shockley partials repel elastically and are pulled together by the **intrinsic** stacking fault between them. They settle where the two forces balance.
($\gamma_{usf}$ is the barrier for *nucleating* a partial. It enters the dislocation core and Peierls stress through the SDVPN model in Section 8, not $d_{eq}$.)

**Isotropic** (Hirth & Lothe), with $\beta$ the angle between $\mathbf b$ and the line direction ($\beta=90°$ edge, $0°$ screw):
$$d_{eq} = \frac{\mu b_p^2}{8\pi\gamma_{isf}}\,\frac{2-\nu}{1-\nu}\left(1-\frac{2\nu\cos2\beta}{2-\nu}\right)$$

**Anisotropic** (Stroh). atomman's `Stroh` solution gives the energy-coefficient tensor $\mathbf K$, so $d_{eq} \approx \dfrac{\mathbf b_{p1}\cdot\mathbf K\cdot\mathbf b_{p2}}{2\pi\gamma_{isf}}$.

In [ ]:
def d_iso(mu_GPa, nu, bp, g_mJ, beta_deg):
    mu_ = uc.set_in_units(mu_GPa, 'GPa'); g_ = uc.set_in_units(g_mJ, 'mJ/m^2')
    beta = np.radians(beta_deg)
    return mu_ * bp**2 / (8 * np.pi * g_) * (2 - nu) / (1 - nu) * (1 - 2 * nu * np.cos(2 * beta) / (2 - nu))

def d_aniso(xi_uvw, slip_hkl=(1, 1, 1)):
    stroh = am.defect.solve_volterra_dislocation(C, b_uvw * a0, ξ_uvw=xi_uvw, slip_hkl=slip_hkl, box=ucell.box)
    T = stroh.transform                              # crystal Cartesian -> dislocation frame
    K_ = stroh.K_tensor
    return (T @ (bp1_uvw * a0)) @ K_ @ (T @ (bp2_uvw * a0)) / (2 * np.pi * uc.set_in_units(gamma_isf, 'mJ/m^2'))

XI = {'edge': ([1, 1, -2], 90.0), 'screw': ([1, -1, 0], 0.0)}
for kind, (xi, beta) in XI.items():
    results[f'd_eq_iso_{kind}']   = d_iso(mu['Hill'], nu, bp, gamma_isf, beta)
    results[f'd_eq_aniso_{kind}'] = d_aniso(xi)
    print(f"{kind:5s}: d_iso = {results[f'd_eq_iso_{kind}']:6.1f} Å   d_aniso = {results[f'd_eq_aniso_{kind}']:6.1f} Å")

## 7. Periodic array of dislocations (PAD)
atomman inserts one edge dislocation ($\mathbf b=\tfrac{a_0}{2}[1\bar10]$, $\boldsymbol\xi=[11\bar2]$, slip plane (111)) using the anisotropic Volterra solution.
The cell is periodic along $x$ ($\mathbf m$, glide direction) and $z$ ($\boldsymbol\xi$), and has free surfaces along $y$ ($\mathbf n$, slip-plane normal).
atomman marks the atoms within `PAD_BOUNDARY` of the $y$ surfaces as atom type 2. They are drawn in a different colour, but the data file passed to LAMMPS has a single atom type,
so the same `potential.mod` works. The script holds the boundary atoms fixed by position instead: everything outside `ylo_free` < y < `yhi_free`.
`base` is the matching perfect crystal, which Section 8 needs as the reference.

**Script contract - `pad.in`**

| in | out |
|---|---|
| `data_file`, `ylo_free`, `yhi_free` [Å] | `E_total` [eV]; the relaxed positions are read from the instance |

The cell is `boundary p s p`, and atoms outside the free slab are held fixed during minimization (optionally followed by a short MD anneal).

In [ ]:
PAD_AMIN, PAD_BMIN, PAD_CMIN = 200.0, 100.0, 10.0     # Å along x (m), y (n), z (ξ)
PAD_BOUNDARY = 10.0                                  # Å of fixed atoms at each y surface

dislocation = am.defect.Dislocation(ucell, C, burgers=b_uvw, ξ_uvw=[1, 1, -2], slip_hkl=[1, 1, 1],
                                    m=[1, 0, 0], n=[0, 1, 0])
base, pad = dislocation.periodicarray(amin=PAD_AMIN, bmin=PAD_BMIN, cmin=PAD_CMIN,
                                      boundarywidth=PAD_BOUNDARY, return_base_system=True)
print('PAD atoms:', pad.natoms, '| fixed (type 2):', int((pad.atoms.atype == 2).sum()))
print(pad.box)
show(pad, titles=['PAD as built: type 1 free, type 2 fixed boundary'], radius=0.6).show()

In [ ]:
free_y = pad.atoms.pos[pad.atoms.atype == 1, 1]
pad_1type = am.System(atoms=am.Atoms(atype=1, pos=pad.atoms.pos), box=pad.box, pbc=pad.pbc, symbols='Ni')
pad_1type.dump('atom_data', f=str(WORK / 'pad.data'))
L = run_lammps('pad.in', POTENTIAL_MOD, {'data_file': WORK / 'pad.data',
                                        'ylo_free': free_y.min() - 0.01, 'yhi_free': free_y.max() + 0.01})
results['E_pad'] = L.extract_variable('E_total')
pad_relaxed = system_from_lammps(L)
L.close()
assert pad_relaxed.natoms == base.natoms

**Before and after relaxation, defect atoms only.** Atoms classed as fcc by CNA, and the fixed boundary atoms, are hidden.
As built, the Volterra solution is a single compact core. After relaxation it splits into two Shockley partials (grey cores) joined by a ribbon of hcp atoms (red),
which is the intrinsic stacking fault from Section 5. The ribbon width is a direct, image-based measure of $d_{eq}$, which can be compared with Section 6
and with HRTEM images of Ni. Rotate the view to look down the line direction $z$.

In [ ]:
cna_pad = {'as built': cna(pad), 'relaxed': cna(pad_relaxed)}
masks = [(c != 1) & (pad.atoms.atype == 1) for c in cna_pad.values()]
show([pad, pad_relaxed], color_by=[np.array([CNA_NAMES[v] for v in c]) for c in cna_pad.values()],
     mask=masks, titles=list(cna_pad)).show()

x_hcp = pad_relaxed.atoms.pos[(cna_pad['relaxed'] == 2) & (pad.atoms.atype == 1), 0]
results['d_eq_PAD_cna'] = np.percentile(x_hcp, 98) - np.percentile(x_hcp, 2) if len(x_hcp) else 0.0
print(f"hcp ribbon width (2nd-98th percentile in x): {results['d_eq_PAD_cna']:.1f} Å")

## 8. Disregistry and CRSS
### 8.1 Atomistic disregistry of the relaxed PAD
$\delta(x) = \mathbf u_{above}(x) - \mathbf u_{below}(x)$ across the slip plane. The two partials show up as two peaks in the dislocation density $\rho = d\delta/dx$, and their spacing is the simulated $d_{eq}$.

In [ ]:
# slip plane: midway between the two (111) planes closest to the dislocation (y = 0 in atomman's PAD)
planepos = np.array([0.0, 0.0, 0.0])
x_md, dis_md = am.defect.disregistry(base, pad_relaxed, m=[1, 0, 0], n=[0, 1, 0], planepos=planepos)

def partial_positions(x, dis):
    rho = np.gradient(dis[:, 0], x)                  # edge component
    order = np.argsort(rho)[::-1]
    p1 = x[order[0]]
    p2 = next(x[i] for i in order[1:] if abs(x[i] - p1) > 2 * bp)   # second peak, not adjacent
    return sorted((p1, p2))

p_md = partial_positions(x_md, dis_md)
results['d_eq_PAD'] = p_md[1] - p_md[0]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x_md, dis_md[:, 0] / b, 'o', ms=3, label=r'$\delta_x$ (edge)')
ax.plot(x_md, dis_md[:, 2] / b, 's', ms=3, label=r'$\delta_z$ (screw)')
for p in p_md: ax.axvline(p, ls=':', c='k')
ax.set(xlabel='x (Å)', ylabel=r'$\delta / |b|$', title=f"PAD: d_eq = {results['d_eq_PAD']:.1f} Å"); ax.legend()

### 8.2 Semidiscrete variational Peierls-Nabarro (SDVPN)
The SDVPN model minimizes the elastic energy plus the misfit energy from the γ-surface (Section 5) on a **discrete** grid. The grid spacing is the
spacing of atomic columns along $\mathbf m$, which is $|\mathbf b|/2$ for this edge dislocation. The discreteness is what produces a finite Peierls stress.

In [ ]:
XSTEP = b / 2
XNUM  = 2 * int(100.0 / XSTEP)                       # grid points (x spans about ±100 Å)

volterra = am.defect.solve_volterra_dislocation(C, b_uvw * a0, ξ_uvw=[1, 1, -2], slip_hkl=[1, 1, 1],
                                                box=ucell.box, m=[1, 0, 0], n=[0, 1, 0])
x, dis0 = am.defect.pn_arctan_disregistry(xnum=XNUM, xstep=XSTEP, burgers=volterra.burgers, halfwidth=3 * bp)

pn = am.defect.SDVPN(volterra=volterra, gamma=gamma, cutofflongrange=XNUM * XSTEP / 2,
                     min_method='L-BFGS-B', min_options={'maxiter': 2000})
pn.solve(x=x, disregistry=dis0)

p_pn = partial_positions(pn.x, pn.disregistry)
results['d_eq_SDVPN'] = p_pn[1] - p_pn[0]
fig = pn.disregistry_plot()
print(f"SDVPN d_eq = {results['d_eq_SDVPN']:.1f} Å   (PAD: {results['d_eq_PAD']:.1f} Å)")

### 8.3 CRSS (Peierls stress) from SDVPN
Apply an increasing resolved shear stress $\tau_{xy}$ (glide direction $\mathbf m$, plane normal $\mathbf n$), re-solve starting from the previous solution each time, and follow the dislocation centre $\bar x = \sum x\rho / \sum\rho$.
The CRSS is the lowest stress at which the centre no longer settles but moves through the grid.

In [ ]:
TAUS_MPa = np.arange(0, 205, 5)
MOVE_TOL = 4 * XSTEP                                 # Å; a displacement this large counts as glide

def centre(x, dis):
    rho = np.abs(np.gradient(dis[:, 0], x))
    return (x * rho).sum() / rho.sum()

x_ref = centre(pn.x, pn.disregistry)
centres, crss = [], None
dis = pn.disregistry.copy()
for t in TAUS_MPa:
    tau = np.zeros((3, 3)); tau[0, 1] = tau[1, 0] = uc.set_in_units(t, 'MPa')
    pn.solve(x=x, disregistry=dis, tau=tau)
    dis = pn.disregistry.copy()
    centres.append(centre(x, dis))
    if abs(centres[-1] - x_ref) > MOVE_TOL:
        crss = t; break

results['CRSS_SDVPN_MPa'] = crss
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(TAUS_MPa[:len(centres)], np.array(centres) - x_ref, 'o-')
ax.set(xlabel=r'$\tau_{xy}$ (MPa)', ylabel=r'$\Delta\bar{x}$ (Å)', title=f'CRSS ≈ {crss} MPa')

## Summary

In [ ]:
out = Path(f'results_{POTENTIAL_NAME}.json')
out.write_text(json.dumps({k: (float(v) if isinstance(v, (np.floating, float, int)) and v is not None else v)
                           for k, v in results.items()}, indent=2))
for k, v in results.items():
    print(f'{k:18s} {v:.4g}' if isinstance(v, (float, np.floating)) else f'{k:18s} {v}')